# CLASE 2 — DESARROLLO FRONTEND PARA APLICACIONES BASADAS EN IA

**Perfil:** estudiantes con lógica de programación básica

**Resultado final:** interfaz web que consume un modelo de IA mediante API REST y gestiona estados reales del modelo

---



## Normas y buenas prácticas que se aplicarán durante toda la clase

| Norma                           | Aplicación en clase                                |
| ------------------------------- | -------------------------------------------------- |
| ISO/IEC 25010                   | Calidad de software (usabilidad, confiabilidad)    |
| OWASP Top 10                    | Validación de entrada y consumo seguro de APIs     |
| W3C HTML5                       | Semántica correcta                                 |
| WCAG 2.1                        | Accesibilidad básica                               |
| ECMAScript 2022                 | JS moderno                                         |
| Arquitectura REST               | Integración con modelo IA                          |
| Clean Code                      | Nombres, funciones pequeñas, responsabilidad única |
| UX for AI Systems (Google PAIR) | Mostrar incertidumbre y estados del modelo         |

---

# OBJETIVO DE LA CLASE

El estudiante comprenderá cómo construir una interfaz web que:

1. capture datos del usuario
2. los envíe a un modelo entrenado
3. interprete la respuesta
4. comunique correctamente la decisión del modelo

> En IA el frontend no es decorativo: es la capa de interpretación humano-modelo.

---




# BLOQUE 1 — ARQUITECTURA DE UNA APLICACIÓN CON MODELO IA (35 min)

## Flujo real en producción

```
Usuario → Frontend → API Backend → Modelo → Postprocesamiento → Frontend → Usuario
```

El frontend debe manejar:

| Problema real       | Consecuencia        |
| ------------------- | ------------------- |
| Latencia del modelo | interfaz bloqueada  |
| Baja confianza      | mala interpretación |
| Error de inferencia | decisión incorrecta |
| Input inválido      | sesgo en predicción |

---



## Principio UX obligatorio en IA

Un modelo NO devuelve verdades
Devuelve probabilidades

Por tanto el frontend debe mostrar:

```
Predicción: Neumonía
Confianza: 72%
```

No:

```
Tiene neumonía
```

---




# BLOQUE 2 — HTML SEMÁNTICO PARA INTERFACES DE IA

## Regla W3C

HTML describe significado, no apariencia.

Construiremos interfaz para clasificador:

Entrada → datos paciente
Salida → diagnóstico + confianza

---

### Estructura base

In [ ]:
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Clasificador Médico IA</title>
</head>

<body>

<main>
    <section aria-labelledby="titulo-modelo">

        <h1 id="titulo-modelo">Sistema de apoyo diagnóstico</h1>

        <form id="prediction-form">

            <label for="temperatura">Temperatura corporal</label>
            <input id="temperatura" type="number" step="0.1" required>

            <label for="tos">Nivel de tos</label>
            <select id="tos" required>
                <option value="">Seleccione</option>
                <option value="0">Ninguna</option>
                <option value="1">Moderada</option>
                <option value="2">Alta</option>
            </select>

            <button type="submit">Evaluar</button>

        </form>

        <output id="resultado" aria-live="polite"></output>

    </section>
</main>

</body>
</html>


---

## Concepto importante

`output` + `aria-live`

Cumple WCAG:
El sistema anuncia cambios del modelo automáticamente a lectores de pantalla.

---



# BLOQUE 3 — CSS FUNCIONAL PARA IA

## Objetivo

Comunicar el estado del modelo

Estados obligatorios en IA:

| Estado      | Significado    |
| ----------- | -------------- |
| idle        | esperando      |
| loading     | inferencia     |
| success     | predicción     |
| uncertainty | baja confianza |
| error       | fallo modelo   |

---



### CSS


In [ ]:
.loading {
    opacity: 0.5;
    pointer-events: none;
}

.success {
    color: #1b5e20;
}

.warning {
    color: #b26a00;
}

.error {
    color: #b00020;
}


---

## Regla UX IA

Nunca ocultar incertidumbre del modelo.

---

# BLOQUE 4 — JAVASCRIPT + API DE MODELO IA (60 min)

## Estructura de petición segura (OWASP)



In [ ]:
async function predict(data) {

    const response = await fetch("http://localhost:8000/predict", {
        method: "POST",
        headers: {
            "Content-Type": "application/json"
        },
        body: JSON.stringify(data)
    });

    if(!response.ok)
        throw new Error("Fallo en inferencia");

    return response.json();
}


---

## Controlador principal




In [ ]:
const form = document.getElementById("prediction-form");
const output = document.getElementById("resultado");

form.addEventListener("submit", async e => {

    e.preventDefault();

    const data = {
        temperatura: parseFloat(temperatura.value),
        tos: parseInt(tos.value)
    };

    output.textContent = "Analizando...";
    output.className = "loading";

    try {
        const result = await predict(data);

        renderResult(result);

    } catch {
        output.textContent = "Error en el modelo";
        output.className = "error";
    }
});


---

## Interpretación responsable del modelo




In [ ]:
function renderResult({label, confidence}) {

    if(confidence < 0.6){
        output.textContent =
          `Resultado incierto (${(confidence*100).toFixed(1)}%)`;
        output.className = "warning";
        return;
    }

    output.textContent =
      `Predicción: ${label} (${(confidence*100).toFixed(1)}%)`;

    output.className = "success";
}


---

# BLOQUE 5 — INTRODUCCIÓN A REACT (35 min)

## Problema del JS tradicional

Manipulación manual del DOM
→ propenso a inconsistencias cuando el modelo responde asincrónicamente

React resuelve:

> La interfaz es función del estado del sistema

---




## Conceptos fundamentales

| Concepto    | Uso en IA           |
| ----------- | ------------------- |
| State       | predicción actual   |
| Props       | datos del modelo    |
| Re-render   | nueva inferencia    |
| Componentes | módulos del sistema |

---

## Ejemplo básico


In [ ]:
```jsx
function Resultado({prediccion}) {
  if(!prediccion) return <p>Esperando datos</p>;

  return (
    <p>
      {prediccion.label} ({prediccion.confidence * 100}%)
    </p>
  );
}
```



---

# BLOQUE 6 — INTERFAZ IA EN REACT

## Estado central


In [ ]:

```jsx
const [prediction, setPrediction] = useState(null);
const [loading, setLoading] = useState(false);
```


---

## Petición al modelo


In [ ]:

```jsx
async function evaluar(data){

  setLoading(true);

  const res = await fetch("http://localhost:8000/predict",{
    method:"POST",
    headers:{"Content-Type":"application/json"},
    body:JSON.stringify(data)
  });

  const result = await res.json();

  setPrediction(result);
  setLoading(false);
}
```




---

## Render declarativo


In [ ]:

```jsx
return (
  <div>
    <Formulario onSubmit={evaluar} disabled={loading}/>
    {loading && <p>Modelo evaluando...</p>}
    <Resultado prediccion={prediction}/>
  </div>
);
```
